# Path Analysis

Deep dive on specific TX→RX propagation paths.

**What you'll learn:**
- Analyze propagation for a specific grid pair
- Hour-by-hour and seasonal profiles
- Compare paths across different bands

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import (
    load_dataset, 
    plot_path_profile,
    grid_to_latlon,
    grid_distance,
    grid_bearing,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
df = load_dataset("wspr")
print(f"Loaded {len(df):,} signatures")

## Define Your Path

Enter the TX and RX grids for your path of interest.

In [ ]:
# Example: Idaho to Central Europe
TX_GRID = "DN13"  # Idaho, USA
RX_GRID = "JN48"  # Central Europe

# Path info
tx_lat, tx_lon = grid_to_latlon(TX_GRID)
rx_lat, rx_lon = grid_to_latlon(RX_GRID)
distance = grid_distance(TX_GRID, RX_GRID)
bearing = grid_bearing(TX_GRID, RX_GRID)

print(f"Path: {TX_GRID} → {RX_GRID}")
print(f"TX: {tx_lat:.2f}°N, {tx_lon:.2f}°")
print(f"RX: {rx_lat:.2f}°N, {rx_lon:.2f}°")
print(f"Distance: {distance:,.0f} km")
print(f"Bearing: {bearing:.1f}°")

In [ ]:
# Filter to this path
# Use first 4 chars for field-level matching
path_df = df[
    (df["tx_grid_4"] == TX_GRID[:4]) & 
    (df["rx_grid_4"] == RX_GRID[:4])
]

print(f"Signatures for this path: {len(path_df):,}")

if len(path_df) == 0:
    print("\nNo exact match. Trying broader search...")
    # Try field-level (2 char)
    path_df = df[
        (df["tx_grid_4"].str[:2] == TX_GRID[:2]) & 
        (df["rx_grid_4"].str[:2] == RX_GRID[:2])
    ]
    print(f"Field-level matches: {len(path_df):,}")

In [ ]:
# Band distribution for this path
BAND_NAMES = {
    102: "160m", 103: "80m", 105: "40m", 107: "20m", 
    109: "15m", 111: "10m"
}

band_counts = path_df.groupby("band").size().sort_index()
print("\nSignatures by band:")
for band, count in band_counts.items():
    name = BAND_NAMES.get(band, str(band))
    print(f"  {name}: {count:,}")

In [ ]:
# Hour profile for 20m
if len(path_df[path_df["band"] == 107]) > 10:
    fig = plot_path_profile(path_df, TX_GRID, RX_GRID, band=107)
    plt.show()
else:
    print("Not enough 20m data for this path")

In [ ]:
# Multi-band hour comparison
fig, ax = plt.subplots(figsize=(14, 6))

for band in [107, 109, 111]:  # 20m, 15m, 10m
    band_df = path_df[path_df["band"] == band]
    if len(band_df) > 10:
        hour_agg = band_df.groupby("hour")["median_snr"].mean()
        ax.plot(hour_agg.index, hour_agg.values, marker="o", 
                label=BAND_NAMES.get(band, str(band)), linewidth=2)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Mean SNR (dB)")
ax.set_title(f"Path: {TX_GRID} → {RX_GRID} — Hour Profile by Band")
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal heatmap for this path
band = 107  # 20m
band_path = path_df[path_df["band"] == band]

if len(band_path) > 50:
    pivot = band_path.groupby(["hour", "month"])["median_snr"].mean().unstack(fill_value=np.nan)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn", center=0, 
                cbar_kws={"label": "SNR (dB)"})
    ax.set_title(f"{TX_GRID} → {RX_GRID} on 20m — Hour × Month")
    ax.set_xlabel("Month")
    ax.set_ylabel("Hour (UTC)")
    plt.tight_layout()
    plt.show()

## Interpretation Tips

**Reading the hour profile:**
- Peak hours = best operating window for this path
- Dead hours = no propagation (MUF below band frequency)
- Consider your local time vs UTC

**Reading the seasonal heatmap:**
- Green = good conditions, Red = poor/no propagation
- Equinox months often have best high-band propagation
- Winter nights = best low-band DX